# PALIMPSEST: Qwen-2.5-Coder-3B CEGIS Training
Optimized for 16GB T4 VRAM, 8-bit AdamW, and Qwen ChatML formatting.

In [ ]:
# Install Unsloth and compatible lightweight fine-tuning dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Import FastLanguageModel and load 4-bit quantized base model
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
# Apply parameter-efficient LoRA adapters targeting all linear attention and MLP projections
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Load dataset and configure Qwen-2.5 ChatML template with ShareGPT mapping
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Configure Qwen-2.5 ChatML template with custom role/content mapping for ShareGPT format
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "from", "content": "value"},
)

# Apply chat template formatting across conversations
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Load local JSONL dataset and map formatting function
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# Initialize 8-bit AdamW SFTTrainer optimized for 16GB T4 GPU VRAM limits
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
# Execute fine-tuning loop
trainer_stats = trainer.train()

In [ ]:
# Enable native 2x faster inference and run a sample MBA verification prompt
FastLanguageModel.for_inference(model)

# Construct sample input query using Qwen-2.5 ChatML format
messages = [
    {"from": "user", "value": "De-obfuscate this 64-bit MBA expression:\n((((x) ^ (y))) + (((2) * (((x) & (y))))))"},
]

# Format prompt and prepare input tensor
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate completion
outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.batch_decode(outputs)
print(response[0])

In [ ]:
# Export model to 4-bit Medium GGUF format for local Ollama and LangGraph execution
model.save_pretrained_gguf("palimpsest_qwen_3b_cegis", tokenizer, quantization_method="q4_k_m")